# 02 — Financial Health Scoring
Uses the production six-dimension scoring implementation and exports one score per company.

In [ ]:
from pathlib import Path
import importlib.util, pandas as pd
root=Path.cwd(); path=root/'etl'/'04_ml_scoring.py'; spec=importlib.util.spec_from_file_location('scoring',path); scoring=importlib.util.module_from_spec(spec); spec.loader.exec_module(scoring)
clean=root/'data'/'clean'
tables={name:pd.read_csv(clean/f'{name.replace("dim_","").replace("fact_","")}.csv') for name in []}
companies=pd.read_csv(clean/'companies.csv'); years=pd.concat([pd.read_csv(clean/f)[['year_label','fiscal_year','quarter','is_ttm','is_half_year','sort_order']] for f in ['profitandloss.csv','balancesheet.csv','cashflow.csv']]).drop_duplicates('year_label').reset_index(drop=True); years['year_id']=years.index+1
def attach(name):
 d=pd.read_csv(clean/f'{name}.csv'); return d.merge(years[['year_id','year_label']],on='year_label',how='left')
tables={'dim_company':companies,'dim_year':years,'fact_profit_loss':attach('profitandloss'),'fact_balance_sheet':attach('balancesheet'),'fact_cash_flow':attach('cashflow'),'fact_analysis':pd.read_csv(clean/'analysis.csv')}
scores=scoring.compute(tables); out=root/'data'/'warehouse'/'ml_scores.csv'; out.parent.mkdir(exist_ok=True); scores.to_csv(out,index=False); display(scores.sort_values('overall_score',ascending=False)); print(out)